In [1]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [1]:
pip install fasttext --no-cache-dir


     ---------------------------------------- 0.0/68.8 kB ? eta -:--:--
     ----- ---------------------------------- 10.2/68.8 kB ? eta -:--:--
     ---------------------- --------------- 41.0/68.8 kB 487.6 kB/s eta 0:00:01
     --------------------------------- ---- 61.4/68.8 kB 544.7 kB/s eta 0:00:01
     -------------------------------------- 68.8/68.8 kB 468.9 kB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for fasttext: filename=fasttext-0.9.2-cp39-cp39-win_amd64.whl size=234274 sha256=aaf18072e266153a7859131012ba2ba3d1d9e968cf08f641810cbc0e57c75d0c
  Stored in directory: C:\Users\shahe\AppData\Local\Temp\pip-ephem-wheel-cache-vbx4wi_4\wheels\64\57\bc\1741406019061d5664914b070bd3e71f6244648732bc96109e
Successfully built fasttext
Note: you may need to restart the kernel to use updated packages.


In [2]:
!pip install fasttext

In [18]:
import pandas as pd
import numpy as np
import json
import fasttext
import re, os, shutil, json
from os import path
from numpy.linalg import norm
import pickle

# Stop Words Processing - for data cleaning
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
sw = set(stopwords.words('english'))
nltk.download('punkt')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\shahe\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\shahe\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [19]:
#drive.mount('/content/gdrive')

defaultPath = "C:\\Users\\shahe\\FYP-PROJECT\\project\\FYP-1\\FYP\\"
Dataset_Folder = defaultPath + "Dataset\\"
Processed_Dataset_Pickle_Path = Dataset_Folder + "processed_dataset.pkl"
Relevant_Comments_List_Pickle_Path = Dataset_Folder + "relevant_comments_list.pkl"
FastTextModelPath = defaultPath + "FastTextModelFile\\cc_en_300-001.bin"


Reading Dataset Files and Merging them

In [20]:
Dataset_Files = os.listdir(Dataset_Folder)

In [21]:
DFs = []
length = 0

for file in Dataset_Files:
  if file.endswith('.csv'):
    df = pd.read_csv(Dataset_Folder + file, usecols=["author", "like_count", "text"])
    DFs.append(df)
    length = length + 1

In [22]:
# Merging DataFrames

dataset_df = pd.concat(DFs)
dataset_df.shape

(1044, 3)

In [23]:
dataset_df.head(5)

,author,like_count,text
0,Adan Phu,0,I actually feel better after going vegan... on...
1,K. Roberts,1,Any time I’ve tried going vegan i overeat beca...
2,Sarar Shleweet,0,"Did not lose weight, did not feel tired, no la..."
3,NIA Mac,0,Feeling tired when you go vegan ??? <br><br>T...
4,Vishal Singh,0,Well I&#39;m a Hindu and non-veg food aren&#39...


In [24]:
ls

 Volume in drive C is Windows
 Volume Serial Number is BA06-CAE6

 Directory of C:\Users\shahe\FYP-PROJECT\project\FYP-1\FYP

05/12/2024  03:18 PM    <DIR>          .
05/11/2024  02:45 AM    <DIR>          ..
05/11/2024  02:52 AM    <DIR>          .ipynb_checkpoints
05/11/2024  02:41 AM     7,237,176,312 cc_en_300-001-001.bin
05/11/2024  02:44 AM    <DIR>          Dataset
05/11/2024  02:44 AM           137,152 Extract_Relevant_Comments and Summarize.ipynb
05/12/2024  03:18 PM           174,632 Extract_Relevant_Comments_and_Summarize (1)-Copy1.ipynb
05/11/2024  01:52 PM    <DIR>          FastTextModelFile
05/11/2024  02:44 AM             7,771 filtering_data.ipynb
05/11/2024  02:44 AM            54,901 Fine_Tuning_Code.ipynb
05/11/2024  02:44 AM            47,839 Fine_Tuning_Code-Check.ipynb
05/11/2024  02:44 AM            16,811 FYP_SUMMARIZATION_MODEL_USE_CASE.ipynb
05/11/2024  02:44 AM            31,368 LOADING DATA AND GENERATING SUMMARY.ipynb
05/12/2024  11:57 AM               272 

Dataset Preprocessing

In [25]:
# Defining Text Preprocing Function

def remove_stop_words(text):
  text_tokens = word_tokenize(text)
  tokens_without_sw = [word for word in text_tokens if not word in sw]
  return ' '.join(map(str, tokens_without_sw))


def preprocess(text):

    text = text.lower()
    text = text.replace("\\n", " ").replace("\n", " ")      # remove \n
    text = remove_stop_words(text)                          # remove stop words
    text = re.sub(r'\b\w{1,2}\b', ' ', text)                # remove words with 1 or 2 characters only
    text = re.sub(r'[^ a-zA-Z0-9]+','', text)                  # remove special characters / remove all except characters and spaces

    if text == "nan" or text == "none":
        text = " "
    text = re.sub(' +', ' ', text)

    return text.strip()

FastText - Features Generation

In [26]:
import fasttext
fastText_model = fasttext.load_model('cc_en_300-001-001.bin')

In [27]:
def fasttext_EmbeddingsGenerator(text):
  text_embeddings = fastText_model.get_word_vector(text)
  return text_embeddings

Sentence Transformer - Features Generation

TF-IDF - Features Generation

In [28]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.pipeline import Pipeline

#print("no. of docs, no. of unique corpus words) : ", tfidf_features_array.shape)
#returns two dimensional feature array against the entire corpus(dataset).
def TFIDF_EmbeddingsGenerator(corpus):
  tfidf_model = TfidfVectorizer(stop_words='english')
  tfidf_model = tfidf_model.fit(corpus) #provide dataset.
  tfidf_features = tfidf_model.transform(corpus) #returns features.
  tfidf_features_array = tfidf_features.toarray() #converts to array.
  tfidf_features_list = []
  for arr in tfidf_features_array:
    tfidf_features_list.append(arr) #appending feature arrays into a list.
  return (tfidf_features_list, tfidf_model)

#generating feature array against the query.
#This function Takes in dataset's feature array and the query.
def TFIDF_QueryFeatureGeneration(tfidf_model, query):
  tfidf_features = tfidf_model.transform([query])
  tfidf_features_array = tfidf_features.toarray()
  return tfidf_features_array[0]

**Creating feature vectors of our Dataset**

In [29]:
# Looping over dataset to generate feature vectors
processed_text = []
fasttext_features = []
tfidf_features = []

for index, row in dataset_df.iterrows():

  processed = preprocess(row["text"]) #sending data for cleaning.
  processed_text.append(processed) #appending in the list.

  # FastText Features Generator
  text_features = fasttext_EmbeddingsGenerator(processed) #takes in one comment at a time and returns features.
  fasttext_features.append(text_features)

  #Sentence Transformer Features Generator
  #
  #

# TFIDF Features Generator
(tfidf_features, tfidf_model) = TFIDF_EmbeddingsGenerator(processed_text) #passing the corpus. The func returns the trained model and extracted features.

dataset_df["processed_text"] = processed_text
dataset_df["fasttext_features"] = fasttext_features
dataset_df["tfidf_features"] = tfidf_features

dataset_df.to_pickle(Processed_Dataset_Pickle_Path, compression='infer', protocol=5, storage_options=None) #saving features data into pickle file.

**Reading Processed Dataset**

In [30]:
dataset_df = pd.read_pickle(Processed_Dataset_Pickle_Path) #reading the file.
dataset_df.head(5)

,author,like_count,text,processed_text,fasttext_features,tfidf_features
0,Adan Phu,0,I actually feel better after going vegan... on...,actually feel better going vegan one drink kac...,"[0.002285124, -0.011500623, -0.0016644135, 0.0...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,K. Roberts,1,Any time I’ve tried going vegan i overeat beca...,time tried going vegan overeat foods feel sati...,"[0.0012658992, -0.007067469, -0.0031459576, 0....","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2,Sarar Shleweet,0,"Did not lose weight, did not feel tired, no la...",lose weight feel tired lack vitamin b12 felt i...,"[0.0028238988, -0.005313926, 0.005609359, 0.01...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
3,NIA Mac,0,Feeling tired when you go vegan ??? <br><br>T...,feeling tired vegan time feel tired get enough...,"[0.0029572814, -0.0057881805, -0.009282095, 0....","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,Vishal Singh,0,Well I&#39;m a Hindu and non-veg food aren&#39...,well hindu nonveg food allowed religion sin ea...,"[0.0013959595, -0.0027295456, -0.0015057367, 0...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


In [31]:
print("fasttext", type(dataset_df["fasttext_features"]))
print("tfidf   ", type(dataset_df["tfidf_features"]))

fasttext <class 'pandas.core.series.Series'>
tfidf    <class 'pandas.core.series.Series'>


**Defining function to extract relevant comments w.r.t query**

In [32]:
def cosine_similarity(A, B):

  # compute cosine similarity to find the similarity between two vectors.Higher the value, higher the similarity.
  cosine = np.dot(A,B)/(norm(A)*norm(B))
  return cosine
def get_relevant_comments_indices(query, comments, top_relevant_comments):
  #finding out relevant comments by returning indexes.
#comments: list of feature vectors.
  cosine_similarities = {}
  index = 0

  for comment in comments:
    cosine = cosine_similarity(query, comment) #calculating cosine similarity between query and the feature vector in feature vector list.
    cosine_similarities[cosine] = index #making a dictionary - key-value pair, cosine is the key and index is value.
    index = index + 1

  # Sort by key
  sorted_cosine_similarities = dict(sorted(cosine_similarities.items(), reverse = True)) #returning a dict of sorted keys(highest cosine value).
  print(sorted_cosine_similarities)
  # create a list of of relevant comments indices
  relevant_indices = []
  for i in range(0, top_relevant_comments): #top_relevant_comments= count.
    key, value = list(sorted_cosine_similarities.items())[i] #accessing top values to get the index of the comment.
    relevant_indices.append(value)

  return relevant_indices

**Ask user query and find relevant comments**

In [33]:
query = input()

# query text preprocessing
processed_query = preprocess(query)

# FastText Features Generator
query_FastText_features = fasttext_EmbeddingsGenerator(processed_query)
# TFIDF Features Generator
query_TFIDF_features = TFIDF_QueryFeatureGeneration(tfidf_model, processed_query)


indices = get_relevant_comments_indices(query_TFIDF_features, dataset_df["tfidf_features"].tolist(), 10) #giving feature vectors of query and the dataset.

print(type(query_TFIDF_features), type(dataset_df["tfidf_features"].tolist()))
print(type(query_TFIDF_features[0]), type(dataset_df["tfidf_features"].tolist()[0]))

# I want to adopt keto diet

 I want to adopt keto


{0.28028433498079: 626, 0.2089522075459834: 264, 0.19922243510089668: 846, 0.18646717677144667: 131, 0.1842336803672835: 968, 0.173471732526749: 606, 0.16400262658266637: 605, 0.1606648798751931: 871, 0.15041898600510326: 976, 0.14628949964161514: 804, 0.14338225311398708: 602, 0.1405938652111038: 380, 0.13770991129163251: 651, 0.13645562765834554: 124, 0.13297799888665718: 789, 0.13028214573723804: 603, 0.13008533356364846: 29, 0.1295484402623706: 916, 0.1282357025326921: 127, 0.11716459303143181: 531, 0.11636686827034678: 79, 0.113757535275481: 38, 0.11356034161988991: 695, 0.11171465540250672: 877, 0.1112494042673343: 625, 0.11025604828634389: 966, 0.10830677118040288: 1027, 0.10827498625737972: 742, 0.10770497236839166: 848, 0.10730366822619139: 957, 0.10682538286793183: 649, 0.10661690928858333: 176, 0.10617872141600135: 220, 0.10558643943115129: 755, 0.10330752023901607: 831, 0.10277928971168203: 835, 0.10210697037042264: 987, 0.10131652598353307: 5, 0.09384843407037836: 878, 0.0

In [25]:
indices

[626, 264, 846, 131, 968, 606, 605, 871, 976, 804]

In [35]:
relevancy_count = 1
relevant_comments_list = []
Comments_str = ""

print("Most Relevant Comments w.r.t Query\n")
for index in indices:
  comment = dataset_df.iloc[index]["text"]
  print(str(relevancy_count) + " relevant comment: " + comment)
  relevancy_count = relevancy_count + 1
  relevant_comments_list.append(comment)
  Comments_str = Comments_str + comment + ". "


with open(Relevant_Comments_List_Pickle_Path, 'wb') as f:
  pickle.dump(relevant_comments_list, f)

Most Relevant Comments w.r.t Query

1 relevant comment: But I don&#39;t want to lose weight
2 relevant comment: What can I do to gain weight? I don’t want to eat garbage but I also want to gain weight
3 relevant comment: I tried KETO ,  83 TO 68
4 relevant comment: What if i dont want too loose weight?
5 relevant comment: sir i am very confused plz help i am 5 &#39;3.5&quot; and weight is 60 i want to increase height and also want muscles
6 relevant comment: Just wondering after the keto out, I&#39;ve learnt that I&#39;d better do the 7 day plan to come back to my normal diet in this video. So can I eat whatever I want to eat and exercise to keep the weight after the 7 day plan that you suggested? Would that maintain my weight? I really can&#39;t follow keto diet forever.. I&#39;d rather eat something I want and exercise to keep my weight.. is that a bad idea? I am on keto diet and this is my 2nd week and it is going pretty good (lost 3.5kg) but I just found from other Youtuber (I shou

In [36]:
print(relevant_comments_list)

['But I don&#39;t want to lose weight', 'What can I do to gain weight? I don’t want to eat garbage but I also want to gain weight', 'I tried KETO ,  83 TO 68', 'What if i dont want too loose weight?', 'sir i am very confused plz help i am 5 &#39;3.5&quot; and weight is 60 i want to increase height and also want muscles', 'Just wondering after the keto out, I&#39;ve learnt that I&#39;d better do the 7 day plan to come back to my normal diet in this video. So can I eat whatever I want to eat and exercise to keep the weight after the 7 day plan that you suggested? Would that maintain my weight? I really can&#39;t follow keto diet forever.. I&#39;d rather eat something I want and exercise to keep my weight.. is that a bad idea? I am on keto diet and this is my 2nd week and it is going pretty good (lost 3.5kg) but I just found from other Youtuber (I should have found that youtuber earlier so this would not have happened), that if I once start keto, it should be my lifestyle otherwise not to

**Text Sumarization**

In [37]:
!pip install evaluate

In [38]:
import nltk
import numpy as np
import pickle
from datasets import load_dataset
from transformers import T5Tokenizer, DataCollatorForSeq2Seq
from transformers import T5ForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer

from datasets import load_dataset
from transformers import AutoModelForSeq2SeqLM
from transformers import AutoTokenizer
from transformers import GenerationConfig

In [5]:
from google.colab import drive
drive.mount('/content/gdrive')

defaultPath = "/content/gdrive/MyDrive/FYP-1/FYP1/"
Dataset_Folder = defaultPath + "Dataset/"
Processed_Dataset_Pickle_Path = Dataset_Folder + "processed_dataset.pkl"
Relevant_Comments_List_Pickle_Path = Dataset_Folder + "relevant_comments_list.pkl"
FastTextModelPath = defaultPath + "FastTextModelFile/cc.en.300-001.bin"


Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


**Read relevant_comments_list**

In [39]:
relevant_comments_list = []

with open(Relevant_Comments_List_Pickle_Path, 'rb') as f:
  relevant_comments_list = pickle.load(f)

#relevant_comments_str = relevant_comments_list[0]
relevant_comments_str = ". ".join(relevant_comments_list) #merging all comments into a single string.


In [40]:
#model_name='facebook/bart-large-cnn'
# model_name = "SyedShaheer/ignore_mode"
model_name='SyedShaheer/distilbart-cnn-12-6_TUNED'
#model_name='SyedShaheer/my_awesome_billsum_model'
#model_name = 'SyedShaheer/SUMMARIZING_STRESSED_MODEL'
# model_name = 'SyedShaheer/pegasus-xsum_tuned'
#model_name = 'SyedShaheer/bart-large-cnn-samsum_tuned'
#model_name = 'SyedShaheer/SUMMARIZING_STRESSED_MODEL'
#model_name = "SyedShaheer/my_awesome_billsum_model2"
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [41]:
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

In [42]:
dialogue = relevant_comments_str

prompt = f"""
Comment:

{dialogue}

Generate a summary?
"""

#generation_config = GenerationConfig(max_new_tokens=10, do_sample=True, temperature=0.5)

inputs = tokenizer(prompt, return_tensors='pt')


output = tokenizer.decode(
    model.generate(
        inputs["input_ids"],
        max_new_tokens=100,
        #generation_config=generation_config,
        )[0],
    skip_special_tokens=True
)

print("-----\n")
print(f'INPUT PROMPT:\n{prompt}')

print("-----\n")
print(f'MODEL GENERATION - \n{output}\n')

-----

INPUT PROMPT:

Comment:

But I don&#39;t want to lose weight. What can I do to gain weight? I don’t want to eat garbage but I also want to gain weight. I tried KETO ,  83 TO 68. What if i dont want too loose weight?. sir i am very confused plz help i am 5 &#39;3.5&quot; and weight is 60 i want to increase height and also want muscles. Just wondering after the keto out, I&#39;ve learnt that I&#39;d better do the 7 day plan to come back to my normal diet in this video. So can I eat whatever I want to eat and exercise to keep the weight after the 7 day plan that you suggested? Would that maintain my weight? I really can&#39;t follow keto diet forever.. I&#39;d rather eat something I want and exercise to keep my weight.. is that a bad idea? I am on keto diet and this is my 2nd week and it is going pretty good (lost 3.5kg) but I just found from other Youtuber (I should have found that youtuber earlier so this would not have happened), that if I once start keto, it should be my lifest

In [34]:
pip install flask

In [36]:
hello = "hello world"

In [48]:
from flask import Flask
app = Flask(__name__)

@app.route('/hello/<name>')
def hello_name(name):
    return 'hello %s' % name

if __name__ == '__main__':
    app.run()


 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [11/May/2024 14:17:26] "GET / HTTP/1.1" 404 -
127.0.0.1 - - [11/May/2024 14:17:30] "GET /hello/shaheer HTTP/1.1" 200 -


In [62]:
from flask import Flask, redirect, url_for, request
app = Flask(__name__)

@app.route('/success/<name>')
def success(name):
    return 'Hello %s!' % name
 
@app.route('/login', methods=['POST','GET'])
def login():
    if request.method == 'POST':
        user = request.form['nm']
        return redirect(url_for('success', name = user))
    else:
        user = request.args.get('nm')
        return redirect(url_for('success', name=user))
        
if __name__ == '__main__':
    app.run()



 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [11/May/2024 21:50:13] "GET / HTTP/1.1" 404 -
[2024-05-11 21:50:17,715] ERROR in app: Exception on /login [GET]
Traceback (most recent call last):
  File "C:\Users\shahe\Anaconda3\envs\torch\lib\site-packages\flask\app.py", line 2525, in wsgi_app
    response = self.full_dispatch_request()
  File "C:\Users\shahe\Anaconda3\envs\torch\lib\site-packages\flask\app.py", line 1822, in full_dispatch_request
    rv = self.handle_user_exception(e)
  File "C:\Users\shahe\Anaconda3\envs\torch\lib\site-packages\flask\app.py", line 1820, in full_dispatch_request
    rv = self.dispatch_request()
  File "C:\Users\shahe\Anaconda3\envs\torch\lib\site-packages\flask\app.py", line 1796, in dispatch_request
    return self.ensure_sync(self.view_functions[rule.endpoint])(**view_args)
  File "C:\Users\shahe\AppData\Local\Temp\ipykernel_10788\1536053298.py", line 15, in login
    return redirect(url_for('success', name=user))
  File "C:\U

In [14]:
from flask import Flask, redirect, url_for, request
app = Flask(__name__)
 
 
@app.route('/success/<name>')
def success(name):
    return 'welcome %s' % name
 
 
@app.route('/login', methods=['POST', 'GET'])
def login():
    if request.method == 'POST':
        user = request.form['nm']
        return redirect(url_for('success', name=user))
    else:
        user = request.args.get('nm')
        return redirect(url_for('success', name=user))


if __name__ == '__main__':
    app.run()

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [12/May/2024 11:56:00] "POST /login HTTP/1.1" 302 -
127.0.0.1 - - [12/May/2024 11:56:00] "GET /login?name=123 HTTP/1.1" 302 -
127.0.0.1 - - [12/May/2024 11:56:00] "GET /login HTTP/1.1" 302 -
127.0.0.1 - - [12/May/2024 11:56:00] "GET /login HTTP/1.1" 302 -
127.0.0.1 - - [12/May/2024 11:56:00] "GET /login HTTP/1.1" 302 -
127.0.0.1 - - [12/May/2024 11:56:01] "GET /login HTTP/1.1" 302 -
127.0.0.1 - - [12/May/2024 11:56:01] "POST /login HTTP/1.1" 302 -
127.0.0.1 - - [12/May/2024 11:56:01] "GET /login?name=123 HTTP/1.1" 302 -
127.0.0.1 - - [12/May/2024 11:56:01] "GET /login HTTP/1.1" 302 -
127.0.0.1 - - [12/May/2024 11:56:01] "POST /login HTTP/1.1" 302 -
127.0.0.1 - - [12/May/2024 11:56:01] "GET /login?name=123 HTTP/1.1" 302 -
127.0.0.1 - - [12/May/2024 11:56:01] "POST /login HTTP/1.1" 302 -
127.0.0.1 - - [12/May/2024 11:56:02] "GET /login?name=123 HTTP/1.1" 302 -
127.0.0.1 - - [12/May/2024 11:56:02] "GET /login HTTP/1.1"

In [13]:
from werkzeug.wrappers import Request, Response
import numpy as np
from flask import Flask, request, render_template
import pickle

app = Flask(__name__, template_folder = 'template')

@app.route('/')
def index():
    return render_template('index')


@app.route('/getprediction',methods=['POST'])
def getprediction():    

    input = request.form.values()
    prediction = input

    return render_template('index.html', output='Predicted Weight in KGs :{}'.format(prediction))
   


if __name__ == "__main__":
    app.run()



 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
[2024-05-12 11:53:55,592] ERROR in app: Exception on / [GET]
Traceback (most recent call last):
  File "C:\Users\shahe\Anaconda3\envs\torch\lib\site-packages\flask\app.py", line 2525, in wsgi_app
    response = self.full_dispatch_request()
  File "C:\Users\shahe\Anaconda3\envs\torch\lib\site-packages\flask\app.py", line 1822, in full_dispatch_request
    rv = self.handle_user_exception(e)
  File "C:\Users\shahe\Anaconda3\envs\torch\lib\site-packages\flask\app.py", line 1820, in full_dispatch_request
    rv = self.dispatch_request()
  File "C:\Users\shahe\Anaconda3\envs\torch\lib\site-packages\flask\app.py", line 1796, in dispatch_request
    return self.ensure_sync(self.view_functions[rule.endpoint])(**view_args)
  File "C:\Users\shahe\AppData\Local\Temp\ipykernel_30940\3726309148.py", line 10, in index
    return render_template('index')
  File "C:\Users\shahe\Anaconda3\envs\torch\lib\site-packages\flask\templating.py", line 146

In [15]:
import numpy as np
from flask import Flask, request

app = Flask(__name__)

@app.route('/')
def index():
    return """
    <!DOCTYPE html>
    <html>
    <head>
        <title>Prediction Page</title>
    </head>
    <body>
        <h1>Enter Input Values</h1>
        <form action="/getprediction" method="post">
            <label for="input_value">Input Value:</label>
            <input type="text" id="input_value" name="input_value"><br><br>
            <input type="submit" value="Submit">
        </form>
    </body>
    </html>
    """

@app.route('/getprediction',methods=['POST'])
def getprediction():    
    input_value = request.form.get('input_value')
    # Perform prediction using input_value here
    prediction = input_value  # Dummy prediction, replace with actual prediction
    
    return 'Predicted Weight in KGs: {}'.format(prediction)

if __name__ == "__main__":
    app.run()


 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [12/May/2024 12:00:18] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [12/May/2024 12:00:21] "POST /getprediction HTTP/1.1" 200 -


In [17]:
import numpy as np
from flask import Flask, request, render_template

app = Flask(__name__)

@app.route('/', methods=['GET', 'POST'])
def index():
    if request.method == 'POST':
        input_value = request.form.get('input_value')
        # Perform prediction using input_value here
        prediction = input_value  # Dummy prediction, replace with actual prediction
        return render_template('index.html', prediction=prediction)
    return render_template('index.html', prediction=None)

if __name__ == "__main__":
    app.run()

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [12/May/2024 12:03:38] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [12/May/2024 12:03:42] "POST / HTTP/1.1" 200 -


In [ ]:
import numpy as np
from flask import Flask, request, render_template

app = Flask(__name__)

@app.route('/', methods=['GET', 'POST'])
def index():
    if request.method == 'POST':
        input_value = request.form.get('input_value')

        # ----------------------------------------------------
        query = input_value

        # query text preprocessing
        processed_query = preprocess(query)

        # FastText Features Generator
        query_FastText_features = fasttext_EmbeddingsGenerator(processed_query)
        # TFIDF Features Generator
        query_TFIDF_features = TFIDF_QueryFeatureGeneration(tfidf_model, processed_query)
        
        
        indices = get_relevant_comments_indices(query_TFIDF_features, dataset_df["tfidf_features"].tolist(), 10) #giving feature vectors of query and the dataset.
        
        print(type(query_TFIDF_features), type(dataset_df["tfidf_features"].tolist()))
        print(type(query_TFIDF_features[0]), type(dataset_df["tfidf_features"].tolist()[0]))


        #--------------------------------------------------
        
        relevancy_count = 1
        relevant_comments_list = []
        Comments_str = ""
        
        print("Most Relevant Comments w.r.t Query\n")
        for index in indices:
          comment = dataset_df.iloc[index]["text"]
          print(str(relevancy_count) + " relevant comment: " + comment)
          relevancy_count = relevancy_count + 1
          relevant_comments_list.append(comment)
          Comments_str = Comments_str + comment + ". "
        
        
        with open(Relevant_Comments_List_Pickle_Path, 'wb') as f:
          pickle.dump(relevant_comments_list, f)  

        #---------------------------------------------------------

        relevant_comments_list = []

        with open(Relevant_Comments_List_Pickle_Path, 'rb') as f:
          relevant_comments_list = pickle.load(f)

        #relevant_comments_str = relevant_comments_list[0]
        relevant_comments_str = ". ".join(relevant_comments_list) #merging all comments into a single string.
        #--------------------------------------------

        model_name='SyedShaheer/distilbart-cnn-12-6_TUNED'
        model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

        #----------------------------------------------

        tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

        #-----------------------------------------------

        dialogue = relevant_comments_str
        
        prompt = f"""
        Comment:
        
        {dialogue}
        
        Generate a summary?
        """
        
        #generation_config = GenerationConfig(max_new_tokens=10, do_sample=True, temperature=0.5)
        
        inputs = tokenizer(prompt, return_tensors='pt')
        
        
        output = tokenizer.decode(
            model.generate(
                inputs["input_ids"],
                max_new_tokens=100,
                #generation_config=generation_config,
                )[0],
            skip_special_tokens=True
        )
        
        print("-----\n")
        print(f'INPUT PROMPT:\n{prompt}')
        
        print("-----\n")
        print(f'MODEL GENERATION - \n{output}\n')
        
        # Perform prediction using input_value here
        prediction = input_value  # Dummy prediction, replace with actual prediction
        return render_template('index.html', prediction=output)
    return render_template('index.html', prediction=None)

if __name__ == "__main__":
    app.run()

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [12/May/2024 16:05:44] "GET / HTTP/1.1" 200 -


{0.7373680911465129: 626, 0.5495423774232141: 86, 0.5365413154608141: 141, 0.5291798752764473: 901, 0.4159450725060595: 254, 0.40593734363480277: 179, 0.38137554012133507: 632, 0.3741044996276017: 170, 0.3504861444633495: 825, 0.3400878214315499: 126, 0.3391823687092825: 704, 0.33736068481973475: 127, 0.3299536816436222: 88, 0.3152543507023387: 717, 0.3058433248296772: 616, 0.29934602077101535: 677, 0.2913888369334931: 670, 0.2851996390351258: 66, 0.2792948598720801: 726, 0.27877595658055687: 603, 0.27777532161988644: 755, 0.27724572653926915: 751, 0.27157934194755595: 656, 0.26388477435031765: 669, 0.26237730460132974: 662, 0.25692285675798515: 695, 0.2509319413221322: 738, 0.2505902450921788: 694, 0.24761993572808774: 95, 0.24446517399154138: 72, 0.24248737171439477: 770, 0.24245315812022616: 739, 0.24066983673001732: 808, 0.23252363911991009: 753, 0.23218935708231075: 799, 0.23051087426124356: 798, 0.22020306641223972: 659, 0.2166532096603969: 789, 0.21644155621343697: 627, 0.216087

127.0.0.1 - - [12/May/2024 16:06:41] "POST / HTTP/1.1" 200 -


-----

INPUT PROMPT:

        Comment:
        
        But I don&#39;t want to lose weight. Lose weight and muscles.. I didn’t really lose weight this wasn’t true for everyone. How to get weight. Lol when people go Vegan just to lose weight, 10 pounds in 10 months is nothing, you could lose 10 pounds in 2 months with exercise and healthy eating.. You will lose weight if you go vegan? Lol I gained 50 pounds 😂. How do you do keto and not lose weight I am already thin trying to get rid of pain.. &quot;On the plus side, expect to lose some weight right away&quot; I&#39;m underweight and it&#39;s difficult for me to achieve a healthier HEAVIER weight. But thanks for presuming everyone wants to lose weight.. Well keto diet is actually the best diet to lose weight bt there are some pro as well as cons. I had done this diet last year for about 2 months only bt i was able to lose that extra 7 kilos.  After losing my weight i was lyk ok now there&#39;s no need to continue this diet i can go for